## byLLM Script

Description here

In [14]:
import os
import pandas as pd
from byllm.lib import Model, by

# This pulls the value from the system variable
apiKey = os.getenv('LLM_API_KEY')

if not apiKey:
    raise ValueError("Missing API Key! Please set the LLM_API_KEY environment variable.")

llm = Model(model_name="gemini/gemini-2.5-flash" , api_key= apiKey)

## Import Datasets

In [16]:
filePath = '../data/fdic/fdic_250.csv'
df = pd.read_csv(filePath)

# df.head()

In [57]:
# Create df2 for a smaller sample in testing
df2 = df.sample(n=25)
# df2.head()

## byLLM model integration

In [47]:
import pandas as pd
import time
from dataclasses import dataclass, asdict

@dataclass
class TextBatch:
    values: list[str]

@by(llm)
def clean_column_values(batch: TextBatch) -> TextBatch:
    """
    Clean this list of strings
    """
    ...

def clean_entire_dataframe(df: pd.DataFrame):
    cleaned_df = df.copy().astype(str)
    
    for col in df.columns:
        print(f"--- Cleaning Column: {col} ---")
        column_data = cleaned_df[col].tolist()
        new_column_values = []
        
        # Process the column in chunks of 50 values (much faster!)
        batch_size = 50 
        for i in range(0, len(column_data), batch_size):
            chunk = column_data[i : i + batch_size]
            
            try:
                # Wrap in dataclass for byllm
                result = clean_column_values(TextBatch(values=chunk))
                new_column_values.extend(result.values)
                
                # Small sleep to respect the 15 RPM limit
                time.sleep(4) 
                
            except Exception as e:
                print(f"Error in {col} at index {i}: {e}")
                new_column_values.extend(chunk) # Keep original on failure
                time.sleep(20)
        
        # Update the column in our dataframe
        # Ensure the lengths match before assignment
        if len(new_column_values) == len(cleaned_df):
            cleaned_df[col] = new_column_values
            
    return cleaned_df

# --- Execution ---
# df_fully_cleaned = clean_entire_dataframe(df2)

## Slice the first 5 columns for rate limitting & Run Model

Current model should be run on a 

In [48]:
df4 = df2.iloc[:, :5]

df_fully_cleaned = clean_entire_dataframe(df4)

--- Cleaning Column: fdic_certificate_number ---

Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

Error in fdic_certificate_number at index 0: litellm.NotFoundError: GeminiException - {
  "error": {
    "code": 404,
    "message": "models/gemini-3.0-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}

--- Cleaning Column: institution_name ---

Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

Error in institution_name a

KeyboardInterrupt: 

In [61]:
#Expected (25, 5)
df_fully_cleaned.shape

(25, 5)